# `meatrider` on a Colab TPU

The tier's TPU program is written but has never run anywhere, because the machine
that wrote it has no TPU. It is a StableHLO module built by
`src/google/tpu/tpu_backend.c`: one SHA-256d hash per lane, the first 64 header
bytes folded into a midstate on the host, both compressions unrolled, and the
target test done by the device as `difficulty` leading zero bits of the
display-order digest.

**Runtime → Change runtime type → TPU**, then *Runtime → Run all*.

What the module looks like before it runs: `python3 src/meatrider/test_meatrider_tpu.py --dump /tmp/mr.hlo`
writes ~857 KB of StableHLO for 8 lanes (12,682 lines, two compressions, one
comparison per 32 bits of difficulty). Compiling that is the expensive part and
it is paid once per challenge.


In [ ]:
# The repo dlopen()s libtpu.so; on Colab's TPU runtime it lives inside the
# Python package of the same name, which is not on the loader's default path.
import glob, os
cands = sorted(
    glob.glob("/usr/local/lib/python3*/dist-packages/libtpu/libtpu.so")
    + glob.glob("/usr/local/lib/python3*/dist-packages/libtpu*/libtpu.so")
    + glob.glob("/usr/share/libtpu*/libtpu.so")
    + glob.glob("/usr/local/lib/libtpu.so"))
print("libtpu candidates:", cands)
if cands:
    os.environ["ANUBIS_TPU_LIB"] = cands[0]
    print("ANUBIS_TPU_LIB =", cands[0])


In [ ]:
%cd /content
!rm -rf anubis-solver
!git clone --depth 1 https://github.com/celikfana-lab/anubis-solver.git
%cd anubis-solver
!git log --oneline -1
!make -j4 2>&1 | tail -6


In [ ]:
# Structure only if no TPU runtime is reachable; structure plus two real mines
# (difficulty 12 and 20) when one is. Every winning digest is re-derived in C.
!python3 src/meatrider/test_meatrider_tpu.py


In [ ]:
# The repo's own TPU self-test: the graceful path without libtpu, then a compile
# and solve if this runtime has one.
!make test-tpu 2>&1 | tail -12


## If the compile takes forever

857 KB of unrolled StableHLO is a lot for XLA. If the first
`anubis_tpu_meatrider_solve()` sits in `PJRT_Client_Compile` for minutes, that is
the compile, not a hang — it is paid once per (midstate, nBits, timestamp), and
the sha256 tier's much smaller module is the reason its TPU run looks fast by
comparison. The number to report is the *second* solve on the same challenge:
the first one includes a compile that the same challenge never pays again.
